# Interpretability: what did the models learn?

We probe the trained checkpoints and use symbolic regression (PySR) to read off the
physical laws each model encodes. The story has two halves:

- **HGNN** factors its dynamics through an explicit scalar potential. We recover it as
  `V(r) = -C/r` (2-body) and, in-distribution from raw distances, `V_total = b - C*sum(1/r)`;
  its physical force is `~1/r^2` at an effective `G ~ 0.86`; and its kinetic energy is an
  emergent-isotropic quadratic `T ~ |v|^2`.
- **EGNN** represents the same dynamics as an entangled coordinate map. Its edge coupling is
  wiggly and sign-changing - there is no clean force law to recover. That contrast is the result.

All heavy lifting lives in `interpretability/`; this notebook just orchestrates and displays.
PySR runs with `progress=True` so you can watch the evolutionary search live.

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from IPython.display import display

from interpretability._io import write_results
from interpretability._types import InterpretabilityResults
from interpretability.analysis import InterpretabilityAnalysis

%matplotlib inline

## Load the trained models

Loads the canonical HGNN and EGNN checkpoints and computes the training distance support.

In [ ]:
analysis = InterpretabilityAnalysis.from_checkpoints(progress=True)
analysis.support_phys, analysis.support_norm

## 1. HGNN potential: `V(r) -> -C/r`

The 2-body probe sweeps a single separation (the transfer probe); the 3-body run regresses the
total potential against the three raw distances. Watch PySR build `b - C*sum(1/r)` from
`+ - * /` at the Pareto knee.

In [ ]:
potential, potential_figs = analysis.recover_potential()
print("2-body knee :", potential.two_body.knee_equation, "   C =", round(potential.two_body_c, 3))
print("3-body knee :", potential.total_symbolic.knee_equation)
print(
    "3-body linear: C =",
    round(potential.total_linear_c, 3),
    " R^2 =",
    round(potential.total_linear_r2, 4),
)
for fig in potential_figs:
    display(fig)

## 2. Physical force law and effective `G`

The recovered potential, in physical units: the force goes as `1/r^2` and the effective
gravitational constant is `~0.86` (vs the simulator's `G = 1`). The forward (v=0) and autograd
gradient probes agree, validating the unit conversion.

In [ ]:
force, force_fig = analysis.physical_force()
print("force exponent:", round(force.force_exponent, 3), " (gravity = -2)")
print(
    "G_eff: forward =", round(force.g_eff_forward, 3), " gradient =", round(force.g_eff_gradient, 3)
)
display(force_fig)

## 3. Node/edge locality (companion finding)

The per-edge readout is a clean function of `r_ij` (high `R^2`) but its shape is not `-1/r`:
the attraction lives in the node channel. Only the total potential is physical, so the internal
`T+V` split is not individually interpretable.

In [ ]:
node_edge, node_edge_fig = analysis.node_edge_locality()
print("alignment R^2:", round(node_edge.alignment_r2, 4))
print(
    "V_node std/|mean|:",
    round(node_edge.vnode_relative_std, 3),
    " corr(V_node, sum 1/r):",
    round(node_edge.vnode_suminv_corr, 3),
)
display(node_edge_fig)

## 4. HGNN kinetic energy: emergent isotropy + quadratic form

The kinetic network is fed raw `(vx, vy)`, so rotational invariance is not built in - yet it
learned `T` as a function of speed alone (circular contours, high isotropy `R^2`), and PySR
recovers the quadratic `T ~ C*(vx^2 + vy^2)`.

In [ ]:
kinetic, kinetic_fig = analysis.recover_kinetic()
print("isotropy R^2:", round(kinetic.isotropy_r2, 4))
print(
    "quadratic coefficient:",
    round(kinetic.quadratic_coefficient, 3),
    " expected:",
    round(kinetic.expected_coefficient, 3),
)
print("kinetic knee:", kinetic.symbolic.knee_equation)
display(kinetic_fig)

## 5. EGNN contrast: no recoverable law

The EGNN layer-0 coupling `w(r)` is wiggly and changes sign (attractive, then repulsive), and
its implied force has no clean `1/r^2` slope. The dynamics are accurate but entangled across
velocity and four layers, so there is no clean target for symbolic regression.

In [ ]:
egnn, egnn_fig = analysis.egnn_contrast()
print("layer-0 |w| exponent:", round(egnn.layer0_weight_exponent, 3), " (gravity = -3)")
print("implied-force exponent:", round(egnn.layer0_force_exponent, 3), " (gravity = -2)")
print("attractive fraction:", round(egnn.attractive_fraction, 2), " (gravity = 1.0)")
display(egnn_fig)

## Save results

Figures were saved per step under `runs/reports/interpretability/figures/`. This writes the
recovered equations and metrics to `results.json`.

In [ ]:
results = InterpretabilityResults(
    potential=potential,
    physical_force=force,
    node_edge=node_edge,
    kinetic=kinetic,
    egnn=egnn,
)
write_results(analysis.results_path, results)
print("wrote", analysis.results_path)